# vLLM integration test

This checks every way AuditKIT uses vLLM, on one open model (`Qwen/Qwen3-1.7B`: a chat model with tool calling and a thinking switch).

| § | Path | What it checks |
|---|---|---|
| 0 | environment | `check_compat(check_cuda=True)`: vLLM ≥ 0.30, transformers 5.x, the GPU, the CUDA builds, and a stale torchaudio |
| 1 | `vllm:` offline engine | generation through the chat template, accuracy and throughput |
| 2 | `vllm:` + `RunConfig.chat_template_kwargs` | Qwen3's `enable_thinking=False` reaches the template: shorter, answer-only replies |
| 3 | `vllm:` + tools | **native** tool calling through the chat template, prompt mode for comparison, and the `agent_eval` harness on the offline engine |
| 4 | lm-eval through vLLM | `run_benchmark` on a loglikelihood task and a generative task |
| 5 | vLLM **OpenAI server** + `api:` | native tool calls (incl. parallel), `chat_template_kwargs` over HTTP, and the `agent_eval` harness verifying a final state |
| 6 | checklist | pass/fail for each item above |

**Runtime:** a Colab **L4 or A100** GPU. vLLM's current wheels need Ampere or newer, and CUDA 13 needs an NVIDIA driver ≥ 580. Linux only.

**Secrets:** `GITHUB_TOKEN`, to install AuditKIT. `HF_TOKEN` is optional; Qwen3 isn't gated.

If the install cell says torch was already imported, restart the session and run from the top. vLLM replaces torch, and a running kernel keeps the old one.

In [ ]:
# ---- settings ----
import os
MODEL = os.environ.get("NB_MODEL", "Qwen/Qwen3-1.7B")
VLLM_ARGS = {"gpu_memory_utilization": 0.80, "max_model_len": 4096}
LIMIT = int(os.environ.get("NB_LIMIT", "50"))       # lm-eval samples per task
PORT = 8000

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at BRANCH (#1's branch, where #10 and every #15 fix are merged) and installs it.
# Locally: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, subprocess, sys
BRANCH = "feat/rag-agent-evals"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ('GITHUB_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass                          # secret not set / not granted to this notebook
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[vllm,lmeval,requests]"], check=True)
    # vLLM >= 0.30 installs a CUDA 13 torch. Colab's preinstalled torchaudio (CUDA 12.8) isn't a vLLM dependency, so pip
    # leaves it, and transformers imports it if present -> "PyTorch and TorchAudio were compiled with different CUDA
    # versions". Nothing here needs audio, so remove it before anything imports torch.
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
    if "torch" in sys.modules:
        raise SystemExit("torch was imported before this install: Runtime -> Restart session, then run from the top.")
    sys.path.insert(0, "/content/AuditKIT-Internal/src")
import auditkit as ak
print("auditkit", ak.__version__, "| secrets:", {k: bool(os.environ.get(k)) for k in ('HF_TOKEN',)})

## 0 · Environment

In [ ]:
import json, subprocess, sys, time
import pandas as pd, requests, torch, transformers
from auditkit import Sample
from auditkit.compat import check_compat
from auditkit.runspec import RunConfig
report = check_compat(check_cuda=True)
print(report)
V = report.versions
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE = "bfloat16" if BF16 else "float16"
print("\ntransformers", transformers.__version__, "| vllm", V.get("vllm"), "| torch", torch.__version__, "| dtype", DTYPE)
assert torch.cuda.is_available(), "vLLM needs a CUDA GPU runtime"

## 1 · `vllm:` offline engine: generation

`VLLMModel` builds a `vllm.LLM` (the kwargs go straight to it), renders each prompt with the model's chat template, and generates. Latency and throughput come from the run's `perf`.

In [ ]:
from auditkit.model.vllm_gen import VLLMModel
QA = [("What is the capital of Australia?", "Canberra"), ("How many sides does a hexagon have?", "6"),
      ("What is 17 times 3?", "51"), ("Which planet is known as the Red Planet?", "Mars"),
      ("Who wrote 'Pride and Prejudice'?", "Jane Austen"), ("What is the chemical symbol for gold?", "Au"),
      ("Quelle est la capitale du Canada ?", "Ottawa"), ("What is the largest ocean on Earth?", "Pacific")]
qa = [Sample(id=f"qa{i}", input="Answer with only the answer. " + q, target=a) for i, (q, a) in enumerate(QA)]
llm = VLLMModel(MODEL, dtype=DTYPE, **VLLM_ARGS)
NO_THINK = RunConfig(max_tokens=64, temperature=0.0, chat_template_kwargs={"enable_thinking": False})
gen = ak.evaluate(qa, model=llm, scorers=["quasi_exact_match", "f1_score"], config=NO_THINK)
print("headline:", {k: round(v, 3) for k, v in gen.headline.items()}, "| errors:", gen.errors)
print("latency p50 (ms):", round(gen.perf["latency_ms"]["p50"], 1), "| throughput:", {k: round(v, 1) for k, v in gen.perf["throughput"].items()})
display(pd.DataFrame([{"q": p.sample_id, "expected": p.expected, "answer": (p.raw_output or "").strip()[:60]} for p in gen.predictions]))

## 2 · `chat_template_kwargs` reaches the vLLM chat template

Qwen3 thinks (`<think>…</think>`) unless its template gets `enable_thinking=False`. `RunConfig.chat_template_kwargs` passes that to every request's template, and it's part of the run fingerprint, so the two runs below never share a cache entry.

In [ ]:
THINK = RunConfig(max_tokens=1024, temperature=0.0)                      # template default: thinking on
think = ak.evaluate(qa[:4], model=llm, scorers=["f1_score"], config=THINK)
no_think = ak.evaluate(qa[:4], model=llm, scorers=["f1_score"], config=RunConfig(max_tokens=1024, temperature=0.0,
                                                                               chat_template_kwargs={"enable_thinking": False}))
def has_reasoning(text):
    return "</think>" in text and bool(text.split("</think>")[0].replace("<think>", "").strip())
def think_share(r):
    return sum(has_reasoning(p.raw_output or "") for p in r.predictions) / len(r.predictions)
mean_len = lambda r: sum(len(p.raw_output or "") for p in r.predictions) / len(r.predictions)
print(f"thinking on : mean reply {mean_len(think):7.0f} chars | replies with reasoning {think_share(think):.0%}")
print(f"thinking off: mean reply {mean_len(no_think):7.0f} chars | replies with reasoning {think_share(no_think):.0%}")
print("fingerprints differ:", think.fingerprint != no_think.fingerprint)
TEMPLATE_KWARGS_OK = mean_len(no_think) < mean_len(think) and think.fingerprint != no_think.fingerprint

## 3 · Tool calling on the offline engine

`vllm:` passes the tool schemas to the model's **own chat template** (`tools=`), as `hf:` does, and parses the calls from the reply. A template that can't render tools is **refused with a `CapabilityError`** that points to prompt mode; the tools are never dropped silently.

**Prompt mode** works on any text model: `ToolCallAdapter(mode="prompt")` writes the schemas into the system message instead. Both are scored below, and then the `agent_eval` harness runs a tool loop on the offline engine.

In [ ]:
from auditkit.errors import CapabilityError
TOOLS = [
    {"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
     "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
TOOL_TASKS = [
    Sample(id="single", input="What's the weather in Lisbon?", tools=TOOLS, expected_tool_calls=[[W("Lisbon")]]),
    Sample(id="parallel", input="What's the weather in Oslo and in Cairo? Call the tool for both at once.", tools=TOOLS,
           expected_tool_calls=[[W("Oslo"), W("Cairo")]]),
    Sample(id="pick_tool", input="What time is it in Seoul?", tools=TOOLS, expected_tool_calls=[[{"name": "get_time", "arguments": {"city": "Seoul"}}]]),
    Sample(id="no_tool", input="What is 2+2? Answer directly, without tools.", tools=TOOLS, expected_tool_calls=[]),
]
TOOL_SCORERS = lambda: [ak.ToolCallF1(), ak.TrajectoryMatch(), ak.ParallelToolCalls(), ak.ToolCallValidity(), ak.ToolPermission()]
TOOL_CFG = RunConfig(max_tokens=256, temperature=0.0, chat_template_kwargs={"enable_thinking": False})
native_offline = ak.evaluate(TOOL_TASKS, model=llm, adapter=ak.ToolCallAdapter(), scorers=TOOL_SCORERS(), config=TOOL_CFG)
print("native tools   :", {k: round(v, 3) for k, v in sorted(native_offline.headline.items())}, "| errors:", native_offline.errors[:2])
prompt_tools = ak.evaluate(TOOL_TASKS, model=llm, adapter=ak.ToolCallAdapter(mode="prompt"), scorers=TOOL_SCORERS(),
                           config=RunConfig(max_tokens=256, temperature=0.0, chat_template_kwargs={"enable_thinking": False}))
print("prompt-mode tools:", {k: round(v, 3) for k, v in sorted(prompt_tools.headline.items())})
for p in native_offline.predictions:
    print(f"  {p.sample_id:10s} {ak.parse_tool_calls(p.raw_output or '')}")

# the agent_eval harness with the offline engine as the policy: it executes the calls and verifies the final state
from auditkit.agent_eval import AgentCase, AgentEvalRunner, AgentEvalSpec, FinalStateAssertion
MOVE_TOOLS = [{"type": "function", "function": {"name": "transfer", "description": "Move money between the user's accounts",
               "parameters": {"type": "object", "properties": {"src": {"type": "string"}, "dst": {"type": "string"},
                              "amount": {"type": "number"}}, "required": ["src", "dst", "amount"]}}}]
class Accounts:
    def __init__(self):
        self.acc = {"checking": 1000.0, "savings": 0.0}
    def __call__(self, name, args):
        a = float(args["amount"]); self.acc[args["src"]] -= a; self.acc[args["dst"]] += a
        return json.dumps({"ok": True, "balances": self.acc})
    def snapshot(self):
        return dict(self.acc)
move = AgentCase(id="move-250", task="Move 250 from checking to savings.", allowed_tools=MOVE_TOOLS,
                 outcome=FinalStateAssertion("savings", equals=250.0))
offline_run = AgentEvalRunner().run(AgentEvalSpec(cases=[move], mode="harness", agent=llm,
                                                  agent_opts={"tool_env": Accounts(), "max_steps": 4},
                                                  config=RunConfig(max_tokens=512, temperature=0.0, chat_template_kwargs={"enable_thinking": False})))
print("harness on vllm:", offline_run.rows[0].outcome["verdict"], "| final state:", offline_run.episodes[0].final_state)
OFFLINE_HARNESS_OK = offline_run.rows[0].outcome["verdict"] == "success"
llm.unload()          # free the GPU for lm-eval's own engine

## 4 · lm-eval through vLLM

`run_benchmark` maps `vllm:<model>` to lm-eval's vLLM backend: `model_args` go to the engine, and `apply_chat_template=True` because this is an instruct model. It returns an AuditKIT `RunResult`.

The tasks are `arc_easy` (loglikelihood) and `gsm8k` (generative).

In [ ]:
bench = ak.run_benchmark(
    ["arc_easy", "gsm8k"], f"vllm:{MODEL}",
    config=RunConfig(limit=LIMIT, num_fewshot=0, seed=0),
    apply_chat_template=True, gen_kwargs="temperature=0,max_gen_toks=384",
    model_args={**VLLM_ARGS, "dtype": DTYPE},
    **({"hf_token": os.environ["HF_TOKEN"]} if os.environ.get("HF_TOKEN") else {}),
)
print("lm-eval via vLLM:", {k: round(v, 3) for k, v in bench.headline.items()})
print("errors:", bench.errors[:3])
import gc; gc.collect(); torch.cuda.empty_cache()

## 5 · A vLLM OpenAI server, evaluated through `api:`

This is how most deployments run vLLM: `vllm serve` with a tool-call parser. `api:` sends the tool schemas natively and reads back structured `message.tool_calls`, so parallel calls are exact turns rather than parsed text.

Three checks against the live server:
1. native tool calling, including a parallel call;
2. `chat_template_kwargs` sent over HTTP;
3. the `agent_eval` harness driving the served model through a real tool loop, then verifying the final state.

In [ ]:
LOG = "/tmp/vllm_server.log"
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--port", str(PORT), "--dtype", DTYPE,
     "--gpu-memory-utilization", str(VLLM_ARGS["gpu_memory_utilization"]), "--max-model-len", str(VLLM_ARGS["max_model_len"]),
     "--enable-auto-tool-choice", "--tool-call-parser", "hermes"],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
deadline = time.time() + 900
while True:
    if server.poll() is not None:
        raise RuntimeError("vLLM server exited:\n" + "".join(open(LOG).readlines()[-40:]))
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).status_code == 200:
            break
    except requests.RequestException:
        pass
    if time.time() > deadline:
        raise TimeoutError("server not healthy in time:\n" + "".join(open(LOG).readlines()[-40:]))
    time.sleep(10)
from auditkit.model.api_gen import APIModel
served = APIModel(MODEL, api_base=f"http://127.0.0.1:{PORT}/v1", api_key="EMPTY", name=f"api:vllm/{MODEL}")
print("server up:", requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=10).json()["data"][0]["id"])

In [ ]:
# 5a · native tool calls through the server's hermes parser
native = ak.evaluate(TOOL_TASKS, model=served, adapter=ak.ToolCallAdapter(), scorers=TOOL_SCORERS(),
                     config=RunConfig(max_tokens=512, temperature=0.0, chat_template_kwargs={"enable_thinking": False}))
print("api: native tools:", {k: round(v, 3) for k, v in sorted(native.headline.items())}, "| errors:", native.errors[:2])
for p in native.predictions:
    turns = ((p.context or {}).get("trace") or {}).get("tool_calls") or []
    print(f"  {p.sample_id:10s} turns={[[(c.get('function') or c)['name'] for c in t] for t in turns]}")

In [ ]:
# 5b · chat_template_kwargs over HTTP: the server applies it to the template
q = [Sample(id="q", input="What is 12 times 12? Answer with the number only.", target="144")]
on = ak.evaluate(q, model=served, scorers=["quasi_exact_match"], config=RunConfig(max_tokens=1024, temperature=0.0))
off = ak.evaluate(q, model=served, scorers=["quasi_exact_match"],
                  config=RunConfig(max_tokens=1024, temperature=0.0, chat_template_kwargs={"enable_thinking": False}))
print(f"thinking on : {len(on.predictions[0].raw_output or ''):5d} chars")
print(f"thinking off: {len(off.predictions[0].raw_output or ''):5d} chars -> {(off.predictions[0].raw_output or '').strip()[:40]!r}")
API_KWARGS_OK = len(off.predictions[0].raw_output or "") < len(on.predictions[0].raw_output or "")

In [ ]:
# 5c · the agent_eval harness: AuditKIT drives the served model through a tool loop and verifies the final state
from auditkit.agent_eval import AgentCase, AgentEvalRunner, AgentEvalSpec, FinalStateAssertion
BANK_TOOLS = [
    {"type": "function", "function": {"name": "transfer", "description": "Move money between the user's accounts",
     "parameters": {"type": "object", "properties": {"src": {"type": "string"}, "dst": {"type": "string"}, "amount": {"type": "number"}},
                    "required": ["src", "dst", "amount"]}}},
    {"type": "function", "function": {"name": "get_balance", "description": "Balance of one account",
     "parameters": {"type": "object", "properties": {"account": {"type": "string"}}, "required": ["account"]}}},
]
class Bank:
    def __init__(self):
        self.acc = {"checking": 1000.0, "savings": 0.0}
    def __call__(self, name, args):
        if name == "get_balance":
            return json.dumps({"balance": self.acc.get(args.get("account"))})
        if name == "transfer":
            a = float(args["amount"]); self.acc[args["src"]] -= a; self.acc[args["dst"]] += a
            return json.dumps({"ok": True, "balances": self.acc})
        return "unknown tool"
    def snapshot(self):
        return dict(self.acc)

case = AgentCase(id="move-250", task="Move 250 from checking to savings, then tell me the new savings balance.",
                 allowed_tools=BANK_TOOLS, outcome=FinalStateAssertion("savings", equals=250.0))
res = AgentEvalRunner().run(AgentEvalSpec(cases=[case], mode="harness", agent=served,
                                          agent_opts={"tool_env": Bank(), "max_steps": 5},
                                          scorers=["tool_call_validity", "agent_loop_detection"],
                                          # the harness forwards generation settings but not chat_template_kwargs,
                                          # so Qwen3 thinks here: give it room to finish before it calls
                                          config=RunConfig(max_tokens=1536, temperature=0.0)))
row, ep = res.rows[0], res.episodes[0]
print("outcome:", row.outcome["verdict"], "| stop:", row.stop_reason, "| final state:", ep.final_state)
print("calls:", [(e.payload.get("function") or e.payload).get("name") for e in ep.tool_call_events()])
HARNESS_OK = row.outcome["verdict"] == "success"

In [ ]:
import os as _os, signal
_os.killpg(server.pid, signal.SIGTERM)          # frees the GPU
server.wait(timeout=60)
print("server stopped")

## 6 · Checklist

In [ ]:
def mm(s):
    return tuple(int(x) for x in (s or "0.0").split(".")[:2])
checks = {
    "vllm >= 0.30 and transformers >= 5.15": mm(V.get("vllm")) >= (0, 30) and mm(V.get("transformers")) >= (5, 15),
    "check_compat: no error-level findings (incl. torchaudio)": report.ok,
    "vllm: generation ran without errors and scored": not gen.errors and bool(gen.headline),
    "vllm: chat_template_kwargs switches thinking off": TEMPLATE_KWARGS_OK,
    "vllm: native tool calls scored": "tool_call_f1" in native_offline.headline and not native_offline.errors,
    "vllm: prompt-mode tool calls scored": "tool_call_f1" in prompt_tools.headline,
    "vllm: harness on the offline engine: verified success": OFFLINE_HARNESS_OK,
    "lm-eval via vLLM produced scores": bool(bench.headline),
    "api: (vLLM server) native tool calls scored": "tool_call_f1" in native.headline and not native.errors,
    "api: chat_template_kwargs reaches the server": API_KWARGS_OK,
    "harness on the served model: verified success": HARNESS_OK,
}
for name, ok in checks.items():
    print(("PASS " if ok else "FAIL ") + name)